### 550. Game Play Analysis IV
```
Table: Activity

+--------------+---------+
| Column Name  | Type    |
+--------------+---------+
| player_id    | int     |
| device_id    | int     |
| event_date   | date    |
| games_played | int     |
+--------------+---------+
(player_id, event_date) is the primary key (combination of columns with unique values) of this table.
This table shows the activity of players of some games.
Each row is a record of a player who logged in and played a number of games (possibly 0) before logging out on someday using some device.
Write a solution to report the fraction of players that logged in again on the day after the day they first logged in, rounded to 2 decimal places. In other words, you need to determine the number of players who logged in on the day immediately following their initial login, and divide it by the number of total players.

The result format is in the following example.

 

Example 1:

Input: 
Activity table:
+-----------+-----------+------------+--------------+
| player_id | device_id | event_date | games_played |
+-----------+-----------+------------+--------------+
| 1         | 2         | 2016-03-01 | 5            |
| 1         | 2         | 2016-03-02 | 6            |
| 2         | 3         | 2017-06-25 | 1            |
| 3         | 1         | 2016-03-02 | 0            |
| 3         | 4         | 2018-07-03 | 5            |
+-----------+-----------+------------+--------------+
Output: 
+-----------+
| fraction  |
+-----------+
| 0.33      |
+-----------+
Explanation: 
Only the player with id 1 logged back in after the first day he had logged in so the answer is 1/3 = 0.33


```

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
from pyspark.sql.functions import col,when,count,round,min,date_add

In [3]:
spark=SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [4]:
Data =[[1, 2, '2016-03-01', 5], [1, 2, '2016-03-02', 6], [2, 3, '2017-06-25', 1], [3, 1, '2016-03-02', 0],
        [3, 4, '2018-07-03', 5]]
Schema=StructType([StructField("player_id",IntegerType(),True),\
                  StructField("device_id",IntegerType(),True),\
                   StructField("event_date",StringType(),True),\
                   StructField("games_played",IntegerType(),True)]
                  
                 )
Activity=spark.createDataFrame(Data,Schema)
Activity=Activity.withColumn("event_date",col("event_date").cast("date"))


In [5]:
Activity.show()

+---------+---------+----------+------------+
|player_id|device_id|event_date|games_played|
+---------+---------+----------+------------+
|        1|        2|2016-03-01|           5|
|        1|        2|2016-03-02|           6|
|        2|        3|2017-06-25|           1|
|        3|        1|2016-03-02|           0|
|        3|        4|2018-07-03|           5|
+---------+---------+----------+------------+



In [6]:
Activity_min=Activity.groupBy('player_id').agg(min("event_date").alias("min_event_date"))
Activity_min.show()                                               

+---------+--------------+
|player_id|min_event_date|
+---------+--------------+
|        1|    2016-03-01|
|        2|    2017-06-25|
|        3|    2016-03-02|
+---------+--------------+



In [7]:
AM=Activity_min.select("*").alias("AM")
AC=Activity.select("player_id","event_date").alias("AC")

In [8]:
AM.join(AC,on=[(col("AM.player_id")==col("AC.player_id")) & (col("AC.event_date")==date_add(col("AM.min_event_date"),1))],how='left').show()

+---------+--------------+---------+----------+
|player_id|min_event_date|player_id|event_date|
+---------+--------------+---------+----------+
|        3|    2016-03-02|     NULL|      NULL|
|        1|    2016-03-01|        1|2016-03-02|
|        2|    2017-06-25|     NULL|      NULL|
+---------+--------------+---------+----------+



In [10]:
result=(
    AM.join(AC,
            on=[(col("AM.player_id")==col("AC.player_id")) & (col("AC.event_date")==date_add(col("AM.min_event_date"),1))]
            ,how='left')
        .select(col("AM.player_id"),col("AM.min_event_date"),col("AC.player_id").alias("ACPlayer_id"),col("AC.event_date"))
)


In [11]:
result.show()

+---------+--------------+-----------+----------+
|player_id|min_event_date|ACPlayer_id|event_date|
+---------+--------------+-----------+----------+
|        3|    2016-03-02|       NULL|      NULL|
|        1|    2016-03-01|          1|2016-03-02|
|        2|    2017-06-25|       NULL|      NULL|
+---------+--------------+-----------+----------+



In [12]:
result.agg(round(count("ACPlayer_id")/count("Player_id"),2).alias("fraction")).show()

+--------+
|fraction|
+--------+
|    0.33|
+--------+



In [13]:
spark.stop()